# AutoML with Endgame

Endgame's AutoML system automates the complete machine learning pipeline through
a **15-stage orchestrator** that handles everything from data profiling to model
explanation. The stages are:

| # | Stage | Description |
|---|-------|-------------|
| 1 | **Profiling** | Compute dataset meta-features (n_samples, n_features, class balance, etc.) |
| 2 | **Quality Guardrails** | Detect data quality issues (leakage, constant columns, high cardinality) |
| 3 | **Data Cleaning** | Handle missing values, remove duplicates, fix dtypes |
| 4 | **Preprocessing** | Encode categoricals, scale numerics, handle text/datetime |
| 5 | **Feature Engineering** | Auto-generate interaction, aggregation, and polynomial features |
| 6 | **Data Augmentation** | SMOTE, ADASYN, or other resampling for imbalanced data |
| 7 | **Model Selection** | Choose candidate models based on meta-features and search strategy |
| 8 | **Model Training** | Train selected models with cross-validation |
| 9 | **Constraint Check** | Verify deployment constraints (latency, memory, model size) |
| 10 | **Hyperparameter Tuning** | Optuna-based HPO on promising models |
| 11 | **Ensembling** | Hill climbing, stacking, or blending of top models |
| 12 | **Threshold Optimization** | Optimize classification decision thresholds |
| 13 | **Calibration** | Platt scaling, isotonic, temperature, or Venn-ABERS calibration |
| 14 | **Post-Training** | Knowledge distillation and conformal prediction |
| 15 | **Explainability** | SHAP feature importances and interaction detection |

Time is automatically allocated across stages based on the chosen **preset**.
This notebook walks through using `TabularPredictor` with different presets
on the Adult Census Income dataset.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from endgame.automl import TabularPredictor, list_presets, get_preset

## 2. Load Data

We use the **Adult Census Income** dataset (OpenML #1590), a binary classification
task predicting whether a person earns more than $50K/year. It has ~48K samples
with a mix of numeric and categorical features, making it a good test for AutoML.

In [ ]:
adult = fetch_openml(data_id=1590, as_frame=True, parser="auto")
df = adult.frame

print(f"Dataset shape: {df.shape}")
print(f"Target column: 'class'")
print(f"\nTarget distribution:")
print(df["class"].value_counts())
print(f"\nFeature types:")
print(df.dtypes.value_counts())
df.head()

In [ ]:
# Create train/test split, keeping the target column in the DataFrame
# TabularPredictor expects the label column to be present in the data
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["class"]
)

print(f"Train samples: {len(train_df)}")
print(f"Test samples:  {len(test_df)}")

## 3. Fast Preset

The `"fast"` preset trains a single LightGBM model with no ensembling,
no hyperparameter tuning, and no feature engineering. It allocates 76% of
the time budget to model training. This is ideal for quick baselines.

In [ ]:
# Show what the fast preset configures
fast_config = get_preset("fast")
print(f"Preset: {fast_config.name}")
print(f"Description: {fast_config.description}")
print(f"Default time limit: {fast_config.default_time_limit}s")
print(f"CV folds: {fast_config.cv_folds}")
print(f"Model pool: {fast_config.model_pool}")
print(f"Ensemble method: {fast_config.ensemble_method}")
print(f"HP tuning: {fast_config.hyperparameter_tune}")
print(f"Feature engineering: {fast_config.feature_engineering}")

In [ ]:
predictor_fast = TabularPredictor(
    label="class",
    presets="fast",
    time_limit=120,  # 2 minutes
    verbosity=2,
)

predictor_fast.fit(train_df)

In [ ]:
# View the model leaderboard
print("Fast Preset Leaderboard:")
predictor_fast.leaderboard_

## 4. Medium Quality Preset

The `"medium_quality"` preset trains multiple models (LightGBM, XGBoost,
CatBoost, Random Forest, Linear) with light feature engineering and
Optuna-based hyperparameter tuning. It also builds an ensemble.

In [ ]:
medium_config = get_preset("medium_quality")
print(f"Preset: {medium_config.name}")
print(f"Description: {medium_config.description}")
print(f"Default time limit: {medium_config.default_time_limit}s")
print(f"CV folds: {medium_config.cv_folds}")
print(f"Model pool: {medium_config.model_pool}")
print(f"Ensemble method: {medium_config.ensemble_method}")
print(f"HP tuning: {medium_config.hyperparameter_tune} ({medium_config.tune_trials} trials)")
print(f"Feature engineering: {medium_config.feature_engineering}")

In [ ]:
# Free memory from the fast predictor before training a new one
del predictor_fast
import gc; gc.collect()

predictor_medium = TabularPredictor(
    label="class",
    presets="medium_quality",
    time_limit=300,  # 5 minutes
    verbosity=2,
)

predictor_medium.fit(train_df)

In [ ]:
# Compare leaderboards
print("Medium Quality Leaderboard:")
print(predictor_medium.leaderboard_)
print(f"\nBest model: {predictor_medium.fit_summary_.best_model}")
print(f"Best CV score: {predictor_medium.fit_summary_.best_score:.4f}")

## 5. Quality Guardrails

Endgame's guardrails system automatically detects data quality issues such as
target leakage, constant/duplicate columns, high-cardinality categoricals, and
class imbalance. With `guardrails_strict=True`, the pipeline will halt on
critical warnings instead of just logging them.

In [ ]:
predictor_strict = TabularPredictor(
    label="class",
    presets="fast",
    time_limit=120,
    guardrails_strict=True,  # Halt on critical quality warnings
    verbosity=2,
)

predictor_strict.fit(train_df)

# Check if any quality warnings were raised
if predictor_strict.report_ and predictor_strict.report_.quality_warnings:
    print(f"\nQuality warnings ({len(predictor_strict.report_.quality_warnings)}):")
    for w in predictor_strict.report_.quality_warnings:
        print(f"  [{w.severity.upper()}] {w.category}: {w.message}")
else:
    print("\nNo quality warnings -- data looks clean.")

## 6. Predictions

Use the best predictor to generate predictions on the held-out test set.
The predictor automatically uses the ensemble (if available) or falls back
to the single best model.

In [ ]:
# Use the medium quality predictor (more models trained)
y_pred = predictor_medium.predict(test_df)

# Extract true labels from the test DataFrame
y_true = test_df["class"].values

acc = accuracy_score(y_true, y_pred)
print(f"Test accuracy: {acc:.4f}")
print()
print(classification_report(y_true, y_pred))

In [ ]:
# Probability predictions are also available
y_proba = predictor_medium.predict_proba(test_df)
print(f"Probability predictions shape: {y_proba.shape}")
print(f"First 5 predictions (P(class=0), P(class=1)):")
print(y_proba[:5])

## 7. AutoML Report

After fitting, `TabularPredictor` generates a structured report with:
- **Summary**: total time, number of models, best score, preset used
- **Stage summary**: per-stage timing and success/failure status
- **Model leaderboard**: all trained models ranked by CV score
- **Quality warnings**: data issues detected by guardrails
- **Feature importances**: SHAP-based importances (if explainability stage ran)
- **Tuning summary**: HPO trial results per model

In [ ]:
report = predictor_medium.report()

# Print the overall summary
print("=" * 60)
print("AutoML Report Summary")
print("=" * 60)
for key, value in report.summary.items():
    print(f"  {key}: {value}")

# Print stage timing
print(f"\nStage Summary:")
if not report.stage_summary.empty:
    print(report.stage_summary.to_string(index=False))

# Print model leaderboard
print(f"\nModel Leaderboard:")
if not report.model_leaderboard.empty:
    print(report.model_leaderboard.to_string(index=False))

# Print feature importances if available
if report.feature_importances is not None and not report.feature_importances.empty:
    print(f"\nTop 10 Features:")
    print(report.feature_importances.head(10).to_string(index=False))

In [ ]:
# The report can also be rendered as markdown
print(report.to_markdown())

## 8. Search Strategies

Endgame supports five search strategies that control how the model space is explored:

| Strategy | Description | Best For |
|----------|-------------|----------|
| `portfolio` | Train a diverse, pre-defined portfolio of models in parallel. This is the default strategy and works well across most datasets. | General use, short time budgets |
| `heuristic` | Data-driven rules select models based on computed meta-features (e.g., dataset size, feature types, class balance). | Medium time budgets, automated pipelines |
| `genetic` | Evolutionary optimization of full pipelines (preprocessing + model + hyperparameters). Uses crossover and mutation to explore the search space. | Long time budgets, maximum quality |
| `random` | Randomly sample valid pipeline configurations. Simple but surprisingly effective as a baseline. | Benchmarking, exploration |
| `bayesian` | Optuna-based Bayesian optimization over model and hyperparameter choices using TPE. | Hyperparameter-sensitive models |

The search strategy is set via the `search_strategy` parameter:

```python
# Use genetic/evolutionary search
predictor = TabularPredictor(
    label="class",
    presets="medium_quality",
    search_strategy="genetic",
    time_limit=600,
)

# Use Bayesian optimization
predictor = TabularPredictor(
    label="class",
    presets="good_quality",
    search_strategy="bayesian",
    time_limit=3600,
)
```

The `"exhaustive"` preset automatically switches to genetic search and uses
the full model pool of 60+ algorithms.

In [ ]:
# List all available presets
print("Available presets:")
for name in list_presets():
    config = get_preset(name)
    tl = f"{config.default_time_limit}s" if config.default_time_limit else "unlimited"
    print(
        f"  {name:20s}  time={tl:>10s}  "
        f"models={len(config.model_pool):>2d}  "
        f"ensemble={config.ensemble_method:>14s}  "
        f"tune={str(config.hyperparameter_tune):>5s}  "
        f"FE={config.feature_engineering}"
    )

## Summary

In this notebook we demonstrated Endgame's AutoML system:

- **`TabularPredictor`** provides a 3-line API for automated machine learning
- The **15-stage pipeline** handles everything from data profiling to model explanation
- **Presets** (`fast`, `medium_quality`, `good_quality`, `high_quality`, `best_quality`, `exhaustive`, `interpretable`) control the speed/quality tradeoff
- **Quality guardrails** detect data issues before training begins
- **Search strategies** (`portfolio`, `heuristic`, `genetic`, `random`, `bayesian`) control model space exploration
- The **report** provides a structured summary of the entire AutoML run

### Next steps

- Try `presets="best_quality"` with a longer time budget for maximum accuracy
- Use `presets="interpretable"` for glass-box models (EBM, RuleFit, MARS, etc.)
- Use `search_strategy="genetic"` or `presets="exhaustive"` for evolutionary pipeline optimization
- Enable `checkpoint_dir="./checkpoints"` for incremental saving during long runs
- Call `predictor.predict_sets(test_df, alpha=0.1)` for conformal prediction intervals
- Call `predictor.refit_full()` to retrain the best model on all data before deployment